# PPO: Qbert (Default Hyperparameters)

**Algorithm:** PPO  
**Environment:** Qbert (`QbertNoFrameskip-v4`)  
**Learning rate:** default (3e-4)  
**Parallel envs:** 8  

In [1]:
# Configuration
ALGO            = "ppo"
ENV_KEY         = "qbert"
TOTAL_TIMESTEPS = 1000000
CHECKPOINT_FREQ = 100000
LEARNING_RATE   = None
RUN_NAME        = None

In [2]:
import os

import ale_py
import gymnasium as gym
from stable_baselines3 import DQN, PPO
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack

gym.register_envs(ale_py)

ENV_IDS = {
    "qbert":      "QbertNoFrameskip-v4",
    "donkeykong": "DonkeyKongNoFrameskip-v4",
}

run_id = f"{ALGO}_{ENV_KEY}" + (f"_{RUN_NAME}" if RUN_NAME else "")
checkpoint_dir = os.path.join("checkpoints", run_id)
os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs("logs", exist_ok=True)

print(f"Run ID      : {run_id}")
print(f"Checkpoints : {checkpoint_dir}")

Run ID      : ppo_qbert
Checkpoints : checkpoints/ppo_qbert


In [3]:
n_envs = 8 if ALGO == "ppo" else 1
vec_env = make_atari_env(ENV_IDS[ENV_KEY], n_envs=n_envs, seed=0)
vec_env = VecFrameStack(vec_env, n_stack=4)
print(f"Environment : {ENV_IDS[ENV_KEY]}  |  n_envs: {n_envs}")

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


Environment : QbertNoFrameskip-v4  |  n_envs: 8


## Fresh Training

Checkpoints saved automatically to `checkpoints/ppo_qbert/` every 100,000 steps.

In [4]:
if ALGO == "dqn":
    kwargs = dict(
        verbose=1, device="cpu", tensorboard_log="logs",
        buffer_size=100_000, learning_starts=10_000,
        batch_size=32, train_freq=4, target_update_interval=1_000,
    )
    if LEARNING_RATE is not None:
        kwargs["learning_rate"] = LEARNING_RATE
    model = DQN("CnnPolicy", vec_env, **kwargs)
elif ALGO == "ppo":
    kwargs = dict(
        verbose=1, device="cpu", tensorboard_log="logs",
        n_steps=128, batch_size=256, n_epochs=4, ent_coef=0.01, clip_range=0.1,
    )
    if LEARNING_RATE is not None:
        kwargs["learning_rate"] = LEARNING_RATE
    model = PPO("CnnPolicy", vec_env, **kwargs)

print(f"Algorithm : {ALGO.upper()}")
print(f"Device    : {model.device}")

Using cpu device
Wrapping the env in a VecTransposeImage.
Algorithm : PPO
Device    : cpu


In [5]:
checkpoint_callback = CheckpointCallback(
    save_freq=max(CHECKPOINT_FREQ // n_envs, 1),
    save_path=checkpoint_dir,
    name_prefix="checkpoint",
    save_replay_buffer=False,
    save_vecnormalize=True,
)

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=checkpoint_callback,
    tb_log_name=run_id,
    log_interval=100,
    progress_bar=True,
    reset_num_timesteps=True,
)

final_path = os.path.join(checkpoint_dir, "final_model")
model.save(final_path)
print(f"\nTraining complete. Final model saved to: {final_path}.zip")

Logging to logs/ppo_qbert_1


Output()

------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 1.4e+03      |
|    ep_rew_mean          | 292          |
| time/                   |              |
|    fps                  | 215          |
|    iterations           | 100          |
|    time_elapsed         | 474          |
|    total_timesteps      | 102400       |
| train/                  |              |
|    approx_kl            | 0.0038543171 |
|    clip_fraction        | 0.117        |
|    clip_range           | 0.1          |
|    entropy_loss         | -1.49        |
|    explained_variance   | 0.766        |
|    learning_rate        | 0.0003       |
|    loss                 | 0.051        |
|    n_updates            | 396          |
|    policy_gradient_loss | -0.000204    |
|    value_loss           | 0.17         |
------------------------------------------


------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 1.55e+03     |
|    ep_rew_mean          | 341          |
| time/                   |              |
|    fps                  | 215          |
|    iterations           | 200          |
|    time_elapsed         | 948          |
|    total_timesteps      | 204800       |
| train/                  |              |
|    approx_kl            | 0.0029015478 |
|    clip_fraction        | 0.139        |
|    clip_range           | 0.1          |
|    entropy_loss         | -1.61        |
|    explained_variance   | 0.872        |
|    learning_rate        | 0.0003       |
|    loss                 | 0.0564       |
|    n_updates            | 796          |
|    policy_gradient_loss | -0.00223     |
|    value_loss           | 0.22         |
------------------------------------------


------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 1.73e+03     |
|    ep_rew_mean          | 432          |
| time/                   |              |
|    fps                  | 215          |
|    iterations           | 300          |
|    time_elapsed         | 1426         |
|    total_timesteps      | 307200       |
| train/                  |              |
|    approx_kl            | 0.0017767868 |
|    clip_fraction        | 0.0747       |
|    clip_range           | 0.1          |
|    entropy_loss         | -1.57        |
|    explained_variance   | 0.927        |
|    learning_rate        | 0.0003       |
|    loss                 | 0.0498       |
|    n_updates            | 1196         |
|    policy_gradient_loss | -0.0015      |
|    value_loss           | 0.129        |
------------------------------------------


...

-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1.86e+03    |
|    ep_rew_mean          | 630         |
| time/                   |             |
|    fps                  | 217         |
|    iterations           | 600         |
|    time_elapsed         | 2830        |
|    total_timesteps      | 614400      |
| train/                  |             |
|    approx_kl            | 0.003049421 |
|    clip_fraction        | 0.115       |
|    clip_range           | 0.1         |
|    entropy_loss         | -1.25       |
|    explained_variance   | 0.944       |
|    learning_rate        | 0.0003      |
|    loss                 | 0.085       |
|    n_updates            | 2396        |
|    policy_gradient_loss | -0.00365    |
|    value_loss           | 0.238       |
-----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 2.01e+03    |
|    ep_rew_mean          | 678         |
| time/                   |             |
|    fps                  | 217         |
|    iterations           | 700         |
|    time_elapsed         | 3292        |
|    total_timesteps      | 716800      |
| train/                  |             |
|    approx_kl            | 0.002849744 |
|    clip_fraction        | 0.12        |
|    clip_range           | 0.1         |
|    entropy_loss         | -1.3        |
|    explained_variance   | 0.986       |
|    learning_rate        | 0.0003      |
|    loss                 | 0.0209      |
|    n_updates            | 2796        |
|    policy_gradient_loss | -0.0073     |
|    value_loss           | 0.0994      |
-----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 2.15e+03    |
|    ep_rew_mean          | 822         |
| time/                   |             |
|    fps                  | 217         |
|    iterations           | 800         |
|    time_elapsed         | 3758        |
|    total_timesteps      | 819200      |
| train/                  |             |
|    approx_kl            | 0.004090038 |
|    clip_fraction        | 0.349       |
|    clip_range           | 0.1         |
|    entropy_loss         | -1.4        |
|    explained_variance   | 0.894       |
|    learning_rate        | 0.0003      |
|    loss                 | 0.0445      |
|    n_updates            | 3196        |
|    policy_gradient_loss | 0.00719     |
|    value_loss           | 0.108       |
-----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 2.43e+03    |
|    ep_rew_mean          | 884         |
| time/                   |             |
|    fps                  | 217         |
|    iterations           | 900         |
|    time_elapsed         | 4231        |
|    total_timesteps      | 921600      |
| train/                  |             |
|    approx_kl            | 0.007284947 |
|    clip_fraction        | 0.18        |
|    clip_range           | 0.1         |
|    entropy_loss         | -1.22       |
|    explained_variance   | 0.838       |
|    learning_rate        | 0.0003      |
|    loss                 | 0.11        |
|    n_updates            | 3596        |
|    policy_gradient_loss | -0.00183    |
|    value_loss           | 0.41        |
-----------------------------------------



Training complete. Final model saved to: checkpoints/ppo_qbert/final_model.zip
